# M0 · Environment check

**Goal:** confirm the SSH tunnel is up, `.env` is resolved, and the staging schema is reachable from this notebook.

**Exit criterion:** this notebook runs end-to-end without error.


## 1. Setup

In [ ]:
# === Setup ===
from __future__ import annotations
import sys, pathlib, os, importlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

for name in ('PIPELINE_BATCH_SIZE', 'PIPELINE_OVERLAP_MINUTES', 'PIPELINE_INCREMENTAL_LOOKBACK_MINUTES'):
    raw = os.environ.get(name)
    if isinstance(raw, str) and '#' in raw:
        os.environ[name] = raw.split('#', 1)[0].strip()

import accent_fleet.config as config_module
config_module = importlib.reload(config_module)
settings = config_module.settings

import accent_fleet.db.engine as db_engine
db_engine = importlib.reload(db_engine)
get_engine = db_engine.get_engine
transaction = db_engine.transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


## 2. Inputs

We will run a handful of read-only sanity queries.

In [ ]:
QUERIES = [
    ("current_user / db",          "SELECT current_user, current_database()"),
    ("staging.path count",         "SELECT COUNT(*) FROM staging.path"),
    ("staging.stop count",         "SELECT COUNT(*) FROM staging.stop"),
    ("staging.rep_overspeed count","SELECT COUNT(*) FROM staging.rep_overspeed"),
    ("staging.notification count", "SELECT COUNT(*) FROM staging.notification"),
    ("staging.rep_activity_daily count", "SELECT COUNT(*) FROM staging.rep_activity_daily"),
    ("warehouse schema exists",    "SELECT 1 FROM information_schema.schemata WHERE schema_name='warehouse'"),
    ("marts schema exists",        "SELECT 1 FROM information_schema.schemata WHERE schema_name='marts'"),
]


## 3. Execute

In [ ]:
import pandas as pd
rows = []
with get_engine().connect() as conn:
    for label, sql in QUERIES:
        try:
            r = conn.execute(text(sql)).first()
            rows.append((label, str(r)))
        except Exception as exc:
            rows.append((label, f"ERROR: {exc}"))
df = pd.DataFrame(rows, columns=["check", "result"])
df


## 4. Inspect

In [ ]:
# Basic assertions — fail loudly if the environment isn't ready.
with get_engine().connect() as conn:
    path_count = conn.execute(text("SELECT COUNT(*) FROM staging.path")).scalar_one()
assert path_count > 0, "staging.path is empty — did the tunnel connect to the right DB?"
print(f"OK — staging.path has {path_count:,} rows.")
